# Build Final Report EDA Figures

Run all cells to reproduce Figures 13-17 from the frozen project data. The notebook writes publication-ready PNG files to `4 Report/Figures/EDA` and does not modify the source CSV files.

In [1]:
from __future__ import annotations

import hashlib
import json
from collections import OrderedDict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.titlesize": 15,
    "axes.labelsize": 11,
})

PROJECT_ROOT_OVERRIDE = None  # Set a Path only if the notebook is launched outside the project tree.

def find_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE is not None:
        candidate = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if (candidate / "00-START-HERE.md").is_file():
            return candidate
        raise FileNotFoundError(f"Invalid PROJECT_ROOT_OVERRIDE: {candidate}")

    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "00-START-HERE.md").is_file() and (candidate / "2 Data Workflow").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate Final Project. Launch Jupyter from the project folder or set PROJECT_ROOT_OVERRIDE."
    )

PROJECT_ROOT = find_project_root()
NEW_MASTER_ROOT = PROJECT_ROOT / "2 Data Workflow" / "Outputs" / "03_NEW_MASTER"
COMMON_MASTER_PATH = NEW_MASTER_ROOT / "exports" / "common_master.csv"
LABEL_AUDIT_PATH = (
    NEW_MASTER_ROOT
    / "rebuild_layers"
    / "1.5 tạo label"
    / "earnings_risk_labels.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "4 Report" / "Figures" / "EDA"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Figure output: {OUTPUT_DIR}")

Project root: C:\Users\Admin\OneDrive\Desktop\Final Project
Figure output: C:\Users\Admin\OneDrive\Desktop\Final Project\4 Report\Figures\EDA


In [2]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for source_path in (COMMON_MASTER_PATH, LABEL_AUDIT_PATH):
    if not source_path.is_file():
        raise FileNotFoundError(source_path)

common = pd.read_csv(COMMON_MASTER_PATH, encoding="utf-8-sig")
labels = pd.read_csv(LABEL_AUDIT_PATH, encoding="utf-8-sig")

required_common = {
    "row_key", "target_year", "risk10_label", "risk10_numeric",
    "total_assets_million_vnd", "customer_loans_million_vnd",
    "customer_deposits_million_vnd", "equity_million_vnd",
    "net_income_million_vnd", "net_interest_income_million_vnd",
    "provision_expense_million_vnd", "roe_pct",
}
required_labels = {"next_quarter_net_income_yoy", "risk10_label"}
missing_common = required_common.difference(common.columns)
missing_labels = required_labels.difference(labels.columns)
if missing_common or missing_labels:
    raise ValueError(f"Missing required columns: common={sorted(missing_common)}, labels={sorted(missing_labels)}")

test = common[pd.to_numeric(common["target_year"], errors="coerce") >= 2024].copy()
test_counts = test["risk10_label"].value_counts()
assert len(common) == 1202, f"Expected 1,202 common-master rows, got {len(common)}"
assert len(test) == 234, f"Expected 234 test rows, got {len(test)}"
assert int(test_counts.get("Risk", 0)) == 40
assert int(test_counts.get("NoRisk", 0)) == 194
assert len(labels) == 1202
assert labels["next_quarter_net_income_yoy"].notna().all()
assert not common["row_key"].duplicated().any()

print(f"common_master: {common.shape[0]:,} rows x {common.shape[1]:,} columns")
print(f"held-out test: {len(test):,} rows | NoRisk={test_counts['NoRisk']} | Risk={test_counts['Risk']}")
print(f"label audit: {len(labels):,} complete next-quarter YoY values")

common_master: 1,202 rows x 373 columns
held-out test: 234 rows | NoRisk=194 | Risk=40
label audit: 1,202 complete next-quarter YoY values


In [3]:
FIGURE_FILES = OrderedDict([
    (13, "Figure_13_Risk_NoRisk_Test_Distribution.png"),
    (14, "Figure_14_Missing_Value_Heatmap.png"),
    (15, "Figure_15_Core_Feature_Correlation.png"),
    (16, "Figure_16_Core_Feature_Boxplots.png"),
    (17, "Figure_17_Net_Income_YoY_Outliers.png"),
])

COLOR_NORISK = "#4C78A8"
COLOR_RISK = "#E45756"
COLOR_NEUTRAL = "#5B8E7D"

def save_figure(fig: plt.Figure, figure_number: int) -> Path:
    path = OUTPUT_DIR / FIGURE_FILES[figure_number]
    fig.savefig(path, bbox_inches="tight", facecolor="white", dpi=200)
    plt.close(fig)
    print(f"Saved Figure {figure_number}: {path.name}")
    return path

## Figure 13 - Risk / NoRisk distribution in the held-out test set

In [4]:
order = ["NoRisk", "Risk"]
counts = test["risk10_label"].value_counts().reindex(order).astype(int)
labels_readable = ["No Earnings Deterioration Risk", "Earnings Deterioration Risk"]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(labels_readable, counts.values, color=[COLOR_NORISK, COLOR_RISK], width=0.58)
for bar, count in zip(bars, counts.values):
    percentage = count / counts.sum() * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + counts.max() * 0.025,
        f"{count} rows\n({percentage:.2f}%)",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )
ax.set_title("Risk / NoRisk Distribution in the Held-Out Test Set")
ax.set_ylabel("Number of bank-quarter observations")
ax.set_xlabel("Risk10 evaluation class")
ax.set_ylim(0, counts.max() * 1.18)
ax.grid(axis="x", visible=False)
sns.despine(ax=ax)
save_figure(fig, 13)

Saved Figure 13: Figure_13_Risk_NoRisk_Test_Distribution.png


WindowsPath('C:/Users/Admin/OneDrive/Desktop/Final Project/4 Report/Figures/EDA/Figure_13_Risk_NoRisk_Test_Distribution.png')

## Figure 14 - Missing-value heatmap by feature family and target year

In [5]:
metadata_columns = {
    "row_key", "ticker", "report_year", "report_quarter", "quarter_index",
    "feature_quarter", "target_year", "target_quarter_number", "target_q_index",
    "target_quarter", "risk10_numeric", "risk10_label", "aux_risk5_numeric",
    "aux_risk5_label", "mg002_distance_weight", "mg035_positive_weight",
}
core_columns = [
    "total_assets_million_vnd", "customer_loans_million_vnd",
    "customer_deposits_million_vnd", "equity_million_vnd",
    "net_income_million_vnd", "net_interest_income_million_vnd",
    "provision_expense_million_vnd", "roe_pct",
]
regular_columns = [column for column in common.columns if column not in metadata_columns]
assigned = set()
feature_families = OrderedDict()

def add_family(name: str, columns: list[str]) -> None:
    selected = [column for column in columns if column in regular_columns and column not in assigned]
    if not selected:
        raise ValueError(f"Feature family is empty: {name}")
    feature_families[name] = selected
    assigned.update(selected)

add_family("Core financial", core_columns)
add_family("ReportNorm expanded", [c for c in regular_columns if c.startswith(("rn__", "rnctx__"))])
add_family("Trajectory / rolling", [c for c in regular_columns if c.startswith("traj__")])
add_family("Same-quarter context", [c for c in regular_columns if c.startswith("ctx__")])
add_family("Robust outlier", [c for c in regular_columns if c.startswith("outlier__")])
add_family("Ticker identity", [c for c in regular_columns if c.startswith("ticker__")])
add_family("Base ratios and changes", [c for c in regular_columns if c not in assigned])
assert assigned == set(regular_columns), "Not every regular feature was assigned to a family"
assert len(regular_columns) == 357

years = sorted(pd.to_numeric(common["target_year"], errors="raise").astype(int).unique())
missing_matrix = pd.DataFrame(index=feature_families.keys(), columns=years, dtype=float)
target_year_numeric = pd.to_numeric(common["target_year"], errors="raise").astype(int)
for family, columns in feature_families.items():
    for year in years:
        missing_matrix.loc[family, year] = (
            common.loc[target_year_numeric.eq(year), columns].isna().to_numpy().mean() * 100
        )

fig, ax = plt.subplots(figsize=(16, 6.5))
sns.heatmap(
    missing_matrix, cmap="YlOrRd", vmin=0, vmax=max(10, float(missing_matrix.max().max())),
    annot=True, fmt=".0f", linewidths=0.4, linecolor="white",
    cbar_kws={"label": "Missing values (%)"}, ax=ax,
)
ax.set_title("Missing Value Rate by Feature Family and Target Year")
ax.set_xlabel("Target year")
ax.set_ylabel("Feature family")
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)
save_figure(fig, 14)

Saved Figure 14: Figure_14_Missing_Value_Heatmap.png


WindowsPath('C:/Users/Admin/OneDrive/Desktop/Final Project/4 Report/Figures/EDA/Figure_14_Missing_Value_Heatmap.png')

## Figure 15 - Spearman correlation matrix of core financial features

In [6]:
display_names = {
    "total_assets_million_vnd": "Total assets",
    "customer_loans_million_vnd": "Customer loans",
    "customer_deposits_million_vnd": "Customer deposits",
    "equity_million_vnd": "Equity",
    "net_income_million_vnd": "Net income",
    "net_interest_income_million_vnd": "Net interest income",
    "provision_expense_million_vnd": "Provision expense",
    "roe_pct": "ROE",
}
correlation = common[core_columns].apply(pd.to_numeric, errors="coerce").corr(method="spearman")
correlation = correlation.rename(index=display_names, columns=display_names)

fig, ax = plt.subplots(figsize=(10.5, 8.5))
sns.heatmap(
    correlation, cmap="RdBu_r", vmin=-1, vmax=1, center=0, square=True,
    annot=True, fmt=".2f", linewidths=0.5, linecolor="white",
    cbar_kws={"label": "Spearman correlation", "shrink": 0.82}, ax=ax,
)
ax.set_title("Spearman Correlation of Core Financial Features")
ax.tick_params(axis="x", rotation=38)
ax.tick_params(axis="y", rotation=0)
save_figure(fig, 15)

Saved Figure 15: Figure_15_Core_Feature_Correlation.png


WindowsPath('C:/Users/Admin/OneDrive/Desktop/Final Project/4 Report/Figures/EDA/Figure_15_Core_Feature_Correlation.png')

## Figure 16 - Boxplots of core financial features on comparable display scales

In [7]:
balance_columns = [
    "total_assets_million_vnd", "customer_loans_million_vnd",
    "customer_deposits_million_vnd", "equity_million_vnd",
]
flow_columns = [
    "net_income_million_vnd", "net_interest_income_million_vnd",
    "provision_expense_million_vnd",
]

balance = common[balance_columns].apply(pd.to_numeric, errors="coerce")
balance = np.log10(balance.where(balance > 0))
balance_long = balance.rename(columns=display_names).melt(var_name="Feature", value_name="Value")

flows = common[flow_columns].apply(pd.to_numeric, errors="coerce")
signed_log_flows = np.sign(flows) * np.log10(1 + flows.abs())
flows_long = signed_log_flows.rename(columns=display_names).melt(var_name="Feature", value_name="Value")
roe = pd.to_numeric(common["roe_pct"], errors="coerce").dropna()

fig, axes = plt.subplots(1, 3, figsize=(18, 6.5), gridspec_kw={"width_ratios": [1.2, 1.1, 0.8]})
flier = {"marker": "o", "markersize": 2.5, "markerfacecolor": COLOR_RISK, "alpha": 0.45, "markeredgewidth": 0}
sns.boxplot(data=balance_long, y="Feature", x="Value", color=COLOR_NORISK, flierprops=flier, ax=axes[0])
axes[0].set_title("Balance-sheet scale")
axes[0].set_xlabel("log10(million VND)")
axes[0].set_ylabel("")

sns.boxplot(data=flows_long, y="Feature", x="Value", color=COLOR_NEUTRAL, flierprops=flier, ax=axes[1])
axes[1].set_title("Income and provision flows")
axes[1].set_xlabel("signed log10(1 + |million VND|)")
axes[1].set_ylabel("")

sns.boxplot(x=roe, color="#F2CF5B", flierprops=flier, ax=axes[2])
axes[2].set_title("Profitability")
axes[2].set_xlabel("ROE (%)")
axes[2].set_ylabel("")

fig.suptitle("Distribution and Outliers of Core Financial Features", fontsize=16, fontweight="bold", y=1.02)
fig.text(0.5, -0.02, "All 1,202 labeled bank-quarter rows; transformations are for display only.", ha="center", fontsize=10)
for ax in axes:
    sns.despine(ax=ax)
fig.tight_layout()
save_figure(fig, 16)

Saved Figure 16: Figure_16_Core_Feature_Boxplots.png


WindowsPath('C:/Users/Admin/OneDrive/Desktop/Final Project/4 Report/Figures/EDA/Figure_16_Core_Feature_Boxplots.png')

## Figure 17 - Outlier review for next-quarter net-income YoY change

In [8]:
yoy_pct = pd.to_numeric(labels["next_quarter_net_income_yoy"], errors="raise") * 100
p01, p99 = yoy_pct.quantile([0.01, 0.99])
q1, median, q3 = yoy_pct.quantile([0.25, 0.50, 0.75])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
iqr_outliers = int(((yoy_pct < lower_fence) | (yoy_pct > upper_fence)).sum())
clipped = yoy_pct.clip(p01, p99)

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={"width_ratios": [1.35, 1]})
axes[0].hist(clipped, bins=42, color=COLOR_NEUTRAL, edgecolor="white", alpha=0.95)
axes[0].axvline(-10, color=COLOR_RISK, linestyle="--", linewidth=2.2, label="Risk10 threshold (-10%)")
axes[0].set_title("Central distribution (1st-99th percentile display)")
axes[0].set_xlabel("Next-quarter net-income YoY change (%)")
axes[0].set_ylabel("Rows")
axes[0].legend(loc="upper right")

sns.boxplot(x=yoy_pct, color="#72B7B2", flierprops={"marker": "o", "markersize": 3, "markerfacecolor": COLOR_RISK, "alpha": 0.5, "markeredgewidth": 0}, ax=axes[1])
axes[1].axvline(-10, color=COLOR_RISK, linestyle="--", linewidth=2.2)
axes[1].set_xscale("symlog", linthresh=10)
axes[1].set_title("Full range (symmetric log scale)")
axes[1].set_xlabel("Next-quarter net-income YoY change (%)")
axes[1].set_yticks([])
stats_text = (
    f"Rows: {len(yoy_pct):,}\n"
    f"Min / max: {yoy_pct.min():.1f}% / {yoy_pct.max():.1f}%\n"
    f"1st / 99th pct: {p01:.1f}% / {p99:.1f}%\n"
    f"Median: {median:.1f}%\n"
    f"IQR outliers: {iqr_outliers:,}"
)
axes[1].text(0.02, 0.96, stats_text, transform=axes[1].transAxes, va="top", ha="left", bbox={"boxstyle": "round,pad=0.5", "facecolor": "white", "edgecolor": "#BBBBBB", "alpha": 0.95})

fig.suptitle("Outlier Review: Next-Quarter Net-Income YoY Change", fontsize=16, fontweight="bold", y=1.02)
fig.text(0.5, -0.02, "Source: frozen earnings_risk_labels.csv. Extreme values are retained; clipping affects only the left-panel display.", ha="center", fontsize=10)
fig.tight_layout()
save_figure(fig, 17)

Saved Figure 17: Figure_17_Net_Income_YoY_Outliers.png


WindowsPath('C:/Users/Admin/OneDrive/Desktop/Final Project/4 Report/Figures/EDA/Figure_17_Net_Income_YoY_Outliers.png')

In [9]:
figure_paths = [OUTPUT_DIR / name for name in FIGURE_FILES.values()]
missing_outputs = [str(path) for path in figure_paths if not path.is_file() or path.stat().st_size == 0]
if missing_outputs:
    raise RuntimeError(f"Missing or empty figure outputs: {missing_outputs}")

audit = {
    "project_root": str(PROJECT_ROOT),
    "sources": {
        "common_master": {"path": str(COMMON_MASTER_PATH.relative_to(PROJECT_ROOT)), "sha256": sha256(COMMON_MASTER_PATH)},
        "label_audit": {"path": str(LABEL_AUDIT_PATH.relative_to(PROJECT_ROOT)), "sha256": sha256(LABEL_AUDIT_PATH)},
    },
    "acceptance": {
        "common_rows": int(len(common)),
        "common_columns": int(common.shape[1]),
        "regular_feature_count": int(len(regular_columns)),
        "test_rows": int(len(test)),
        "test_risk": int(test_counts["Risk"]),
        "test_norisk": int(test_counts["NoRisk"]),
        "label_audit_rows": int(len(labels)),
        "label_audit_yoy_missing": int(labels["next_quarter_net_income_yoy"].isna().sum()),
    },
    "figures": [
        {"figure": number, "file": name, "sha256": sha256(OUTPUT_DIR / name), "bytes": (OUTPUT_DIR / name).stat().st_size}
        for number, name in FIGURE_FILES.items()
    ],
}
audit_path = OUTPUT_DIR / "FIGURE_GENERATION_AUDIT.json"
audit_path.write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")

print("\nGeneration complete.")
print(f"Created {len(figure_paths)} PNG files and {audit_path.name}.")
pd.DataFrame(audit["figures"])[["figure", "file", "bytes"]]


Generation complete.
Created 5 PNG files and FIGURE_GENERATION_AUDIT.json.


,figure,file,bytes
0,13,Figure_13_Risk_NoRisk_Test_Distribution.png,71354
1,14,Figure_14_Missing_Value_Heatmap.png,163938
2,15,Figure_15_Core_Feature_Correlation.png,256232
3,16,Figure_16_Core_Feature_Boxplots.png,130444
4,17,Figure_17_Net_Income_YoY_Outliers.png,176800
